
<h1>Prédire les cocnepts Rameau avec une base de vecteurs</h1>


* Pour chaque ligne de l'export, tu calcules le vecteur du titre-résumé. Rappel : une ligne peut posséder plusieurs vedettes rameau  (une vedette, c'est une zone 606, avec ou sans subdivision, cad avec un ou plusieurs ppn)
* Puis, pour chaque vedette rameau, tu agrèges les titres+résumés associés (GROUP BY), et tu fais la moyenne de leur vecteur (AGG(MEAN)
* Tu obtiens donc un vecteur par vedette rameau, qui "représente" toute la richesse contenue dans les titres et résumés des notices biblio ayant cette vedette. Appelons-ça ta grosse matrice Rameau++
* Ensuite, avec ce modèle, tu peux prédire les vedettes à associer à une nouvelle notice. Pour ce faire, tu vectorises le titre+résumé de la nouvelle notice et tu cherches le vecteur le plus proche de ta grosse matrice Rameau++



In [ ]:
#test d'extraction rameau a l'aide d'une base de vecteurs
import sys
!{sys.executable} -m pip install pinecone-client
!{sys.executable} -m  install sentence-transformers
!{sys.executable} -m  install datasets


In [1]:
import pandas as pd
import numpy as np
import simplemma
import nltk
from texthero import preprocessing as preprocessing2
import texthero as hero
import pickle


def lemmatize_text(text):
    w_tokenizer = nltk.tokenize.WhitespaceTokenizer()
    return ' '.join(word for word in [simplemma.lemmatize(w, lang='fr') for w in w_tokenizer.tokenize(text)])

2023-12-12 15:55:38.865264: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: SSE4.1 SSE4.2 AVX AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
/srv/Annif/anaconda3/lib/python3.8/site-packages/spacy/util.py:910: UserWarning: [W095] Model 'en_core_web_sm' (3.2.0) was trained with spaCy v3.2.0 and may not be 100% compatible with the current version (3.6.1). If you see errors or degraded performance, download a newer compatible model or retrain your custom model with the current spaCy version. For more details and available updates, run: python -m spacy validate
  warnings.warn(warn_msg)


In [235]:
import pinecone

# connect to pinecone environment
pinecone.init(
    api_key="VOTRE_CLE_PINECONE",
    environment="us-west4-gcp"  # find next to API key in console
)

In [ ]:
if index_name in pinecone.list_indexes():
    pinecone.delete_index(index_name)

In [236]:
# pour re-indexer il faut au prealable supprimer l'index sur l'interface d'admin
index_name = 'extreme-ml'

# check if the extreme-ml index exists
if index_name not in pinecone.list_indexes():
    # create the index if it does not exist
    pinecone.create_index(
        index_name,
        dimension=384,
        
        metric="cosine"
    )

# connect to extreme-ml index we created
index = pinecone.Index(index_name)

In [ ]:
#chargemnt fichier d'entrainement

#df1 = pd.read_csv('export_picone.csv', sep='\t', skiprows=[12635,14570,20335,20655,32592,32975,32976,33310,50886,58789,58841,
#59676,63232,67341,72847,79427,80517,88343])
df1 = pd.read_csv('working_data_sans_dewey.csv', sep=',')

a = ['000308838','003632806','047450037','058296182','059911174','067313493','070072973','076503909','076986152','077463560','077880560','086077368','103220844','120997703','126056536','137422091','146527313','146979168','147294509','157175065','159761875','162374631','163093741','166049921','166278351','176553460','181543656','182508188','183201523','191351059','191415782','191552208','192576445','192816969','196109590','196122708','197101267','198384122','198388810','200050818','200404342','201602423','219465118','221455183','223827959','225697122','227065069','230373828','230756883','231055099','231860838','232821909','234544538','235109614','235130273','235280011','235755265','236616587','236660640','237131560','237156989','241152550','243051646','248194305','248590413','248915053','248944479','249549492','252457234','252816528','254162525','254992609','255264887','257349006','257504990','257936432','258740043','261199609','261643614','262267888','262760606','263439038','263487784','263926400','265476585','266197809','267884575','268799458','268924759','00094758X','05224170X','11707764X','18171681X','19580547X','23097368X','23690454X','24155859X','25561280X','26117309X','26753177X']
df1.info(verbose = False)
exclus=df1[df1['PPN'].isin(a)]
df1 = df1[~df1['PPN'].isin(a)]

df1.info(verbose = False)

mask = np.random.rand(len(df1)) < 0.81
training_data = df1[mask]
training_data.info(verbose = False)
training_data['PPN'].to_csv('data/training_data.csv',index=False)
testing_data = df1[~mask]
testing_data.info(verbose = False)
testing_data = testing_data.append(exclus, ignore_index=True)
testing_data.info(verbose = False)
testing_data['PPN'].to_csv('data/test_data.csv',index=False)
training_data["RESUME"] = training_data['TITRE'].astype(str) +", "+ training_data["RESUME"]
df=training_data.drop(columns=['PPN','TITRE','DEWEY'])

df['RAMEAU']=df['RAMEAU'].str.replace('_','#u#', regex=True).replace('"','#d#', regex=True).replace('\'','#c#', regex=True).replace(' ','_', regex=True)

#df.drop(index=df.index[89000:],axis=0, inplace=True)
#df.reset_index(drop=True, inplace=True)
#tronque
df['RESUME'] = df['RESUME'].str.slice(0,1000)

w_tokenizer = nltk.tokenize.WhitespaceTokenizer()
lemmatizer = nltk.stem.WordNetLemmatizer()
#lemmatization
df['RESUME'] = df['RESUME'].apply(lemmatize_text)

df.reset_index(drop=True, inplace=True)


In [257]:
testing_data.to_csv('testing_data_full.csv',index=False)

In [225]:

#NETTOYAGE
clean_pipeline = [preprocessing2.fillna,
                   preprocessing2.lowercase,
                   preprocessing2.remove_whitespace,
                   preprocessing2.remove_diacritics
                   
                  ]
df['RESUME'] = hero.clean(df['RESUME'], clean_pipeline)

df.head(5)

,CLE,RESUME,RAMEAU,DESCR,presence_chaine_indexation,rameau_chaines_index,rameau_concepts
0,0,"le culture pour vivre, mort de le culture popu...",Culture_populaire;Diffusion_de_la_culture;Poli...,La culture pour vivre Mort de la culture popul...,False,"['Culture populaire', 'Diffusion de la culture...","['Culture populaire', 'Diffusion de la culture..."
1,1,"le nuit, le jour : essai psychanalytique sur l...",Complexe_de_castration;Psychanalyse;Rêves,"La nuit, le jour : essai psychanalytique sur l...",False,"['Complexe de castration', 'Psychanalyse', 'Rê...","['Complexe de castration', 'Psychanalyse', 'Rê..."
2,2,"ruptures, cultures, il falloir imaginer robins...",Culture,"Ruptures, cultures Il faut imaginer Robinson s...",False,['Culture'],['Culture']
3,4,"le destruction de le temple, oswald tire sur k...",Science-fiction_américaine_--_Traductions_fran...,La Destruction du temple Oswald tire sur Kenne...,True,['Science-fiction américaine -- Traductions fr...,"['Science-fiction américaine', 'Traductions fr..."
4,5,"son pere jove, pelerin de l'image, naturalise ...",Photographie,"Mon père Jové, pèlerin de l'image Naturalisé f...",False,['Photographie'],['Photographie']


In [226]:

df['RAMEAU'] = df['RAMEAU'].str.split(';')
df.head(5)

,CLE,RESUME,RAMEAU,DESCR,presence_chaine_indexation,rameau_chaines_index,rameau_concepts
0,0,"le culture pour vivre, mort de le culture popu...","[Culture_populaire, Diffusion_de_la_culture, P...",La culture pour vivre Mort de la culture popul...,False,"['Culture populaire', 'Diffusion de la culture...","['Culture populaire', 'Diffusion de la culture..."
1,1,"le nuit, le jour : essai psychanalytique sur l...","[Complexe_de_castration, Psychanalyse, Rêves]","La nuit, le jour : essai psychanalytique sur l...",False,"['Complexe de castration', 'Psychanalyse', 'Rê...","['Complexe de castration', 'Psychanalyse', 'Rê..."
2,2,"ruptures, cultures, il falloir imaginer robins...",[Culture],"Ruptures, cultures Il faut imaginer Robinson s...",False,['Culture'],['Culture']
3,4,"le destruction de le temple, oswald tire sur k...",[Science-fiction_américaine_--_Traductions_fra...,La Destruction du temple Oswald tire sur Kenne...,True,['Science-fiction américaine -- Traductions fr...,"['Science-fiction américaine', 'Traductions fr..."
4,5,"son pere jove, pelerin de l'image, naturalise ...",[Photographie],"Mon père Jové, pèlerin de l'image Naturalisé f...",False,['Photographie'],['Photographie']


In [ ]:
#ne marche pas apres car type string suppression de ce qui suit -- 
#import re
#ch = '_--_'
#pattern  = ch + ".*"
#df["RAMEAU"] = df["RAMEAU"].apply(lambda x: re.sub(pattern, '', str(x) ))
#df.head(5)


In [264]:
#inutile
#creation jeux de test (les 460 derniers)
df_test=df.copy()
df_test.drop(index=df_test.index[:89000],axis=0, inplace=True)
df_test.reset_index(drop=True, inplace=True)
df_test
#print (len(df_test))

,RESUME,RAMEAU
0,"droit contemporain un pays arabes, souvent reg...",[Droit_islamique_--_Histoire]
1,"365 personnage qui avoir faire l'alsace, l'obj...",[Célébrités]
2,le guide de le trail : un premier foulee a les...,"[Course_en_montagne, Courses_à_pied, Raids_spo..."
3,"le savoir rural de le moyen age a son jours, l...","[Conditions_rurales_--_Histoire, Patrimoine_ru..."
4,"attentat de bassam : dans le feu de l'action, ...","[Terrorisme, Terrorisme_--_Politique_publique]"
...,...,...
411,le sentiment d'humanite : manifeste pour un fe...,"[Humanité, Réalisation_de_soi]"
412,"le saga un trois petit pois, trois pois, real,...","[Aliments_--_Ouvrages_pour_la_jeunesse, Pois_(..."
413,"droit de le representation de le personnel, pr...",[Représentation_du_personnel]
414,droit penal social : droit penal de le travail...,"[Droit_pénal, Sécurité_sociale_--_Droit_--_Dis..."


In [265]:
#inutile
#creation jeu de vectorization, les 89000 premiers
#suppression records de test
#index_list = list(range(9000, 9416))
#df.drop(df.index[index_list], inplace =True)
df.drop(index=df.index[89000:],axis=0, inplace=True)
df.reset_index(drop=True, inplace=True)
print (len(df))
df

89000


,RESUME,RAMEAU
0,"le culture pour vivre, mort de le culture popu...","[Culture_populaire, Diffusion_de_la_culture, P..."
1,"le nuit, le jour : essai psychanalytique sur l...","[Complexe_de_castration, Psychanalyse, Rêves]"
2,"ruptures, cultures, il falloir imaginer robins...",[Culture]
3,"le revolution structurale, mutation ou crises,...",[Structuralisme]
4,"le destruction de le temple, oswald tire sur k...",[Science-fiction_américaine_--_Traductions_fra...
...,...,...
88995,"le violence de l'etat, presentation un differe...","[Violence_policière, Violence_politique, État]"
88996,l'etat en france : entre deconstruction et rei...,"[Souveraineté, État, État_providence]"
88997,"le revolution militaire napoleonienne, apres a...","[Art_et_science_militaires, Art_et_science_mil..."
88998,franchise et partenariat : developper ou integ...,"[Alliances_stratégiques_(affaires), Concession..."


In [227]:
#chargement model
from sentence_transformers import SentenceTransformer
import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'

# load the model from huggingface
model = SentenceTransformer(
    'sentence-transformers/all-MiniLM-L6-v2',
    device=device
)
model

SentenceTransformer(
  (0): Transformer({'max_seq_length': 256, 'do_lower_case': False}) with Transformer model: BertModel 
  (1): Pooling({'word_embedding_dimension': 384, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': True, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False})
  (2): Normalize()
)

In [228]:
df

,CLE,RESUME,RAMEAU,DESCR,presence_chaine_indexation,rameau_chaines_index,rameau_concepts
0,0,"le culture pour vivre, mort de le culture popu...","[Culture_populaire, Diffusion_de_la_culture, P...",La culture pour vivre Mort de la culture popul...,False,"['Culture populaire', 'Diffusion de la culture...","['Culture populaire', 'Diffusion de la culture..."
1,1,"le nuit, le jour : essai psychanalytique sur l...","[Complexe_de_castration, Psychanalyse, Rêves]","La nuit, le jour : essai psychanalytique sur l...",False,"['Complexe de castration', 'Psychanalyse', 'Rê...","['Complexe de castration', 'Psychanalyse', 'Rê..."
2,2,"ruptures, cultures, il falloir imaginer robins...",[Culture],"Ruptures, cultures Il faut imaginer Robinson s...",False,['Culture'],['Culture']
3,4,"le destruction de le temple, oswald tire sur k...",[Science-fiction_américaine_--_Traductions_fra...,La Destruction du temple Oswald tire sur Kenne...,True,['Science-fiction américaine -- Traductions fr...,"['Science-fiction américaine', 'Traductions fr..."
4,5,"son pere jove, pelerin de l'image, naturalise ...",[Photographie],"Mon père Jové, pèlerin de l'image Naturalisé f...",False,['Photographie'],['Photographie']
...,...,...,...,...,...,...,...
125259,169658,blablabla : en finir avec le bavardage climati...,"[Réchauffement_de_la_Terre, Écologie]",Blablabla : en finir avec le bavardage climati...,False,"['Réchauffement de la Terre', 'Écologie']","['Réchauffement de la Terre', 'Écologie']"
125260,169659,politique de transition ecologique : democrati...,"[Aménagement_du_territoire, Marchés_publics, T...",Politique de transition écologique : Démocrati...,True,"['Aménagement du territoire', 'Marchés publics...","['Aménagement du territoire', 'Marchés publics..."
125261,169660,"abecedaire : mot et rite d'ailleurs, je dedier...","[Ethnopsychiatrie, Psychanalyse_et_ésotérisme,...",Abécédaire : mots et rites d'ailleurs Je dédie...,False,"['Ethnopsychiatrie', 'Psychanalyse et ésotéris...","['Ethnopsychiatrie', 'Psychanalyse et ésotéris..."
125262,169661,"consommer moins, consommer mieux, un cahier pr...","[Biens_de_consommation_durables, Consommation_...","Consommez moins, consommez mieux Un cahier pra...",True,"['Biens de consommation durables', 'Consommati...","['Biens de consommation durables', 'Consommati..."


In [229]:
#creation des vecteurs
import pandas as pd

# Create embeddings
encoded_articles = model.encode(df['RESUME'].tolist(), show_progress_bar=True)
# add the embeddings to our dataframe
df['content_vector'] = pd.Series(encoded_articles.tolist())

Batches:   0%|          | 0/3915 [00:00<?, ?it/s]

In [230]:
#agregation
import numpy as np


df_explode = df.explode('RAMEAU')
#df_explode.head(5)

label_vectors = df_explode.groupby('RAMEAU').agg(mean=('content_vector', lambda x: np.vstack(x).mean(axis=0).tolist()))
label_vectors['target'] = label_vectors.index
label_vectors.columns = ['content_vector', 'label']

label_vectors.sample(10)



,content_vector,label
RAMEAU,,
Préfets_de_région,"[-0.06549372524023056, 0.03866653889417648, -0...",Préfets_de_région
Vitraux_--_Cathédrale_Notre-Dame,"[-0.0010548313148319721, 0.09218093007802963, ...",Vitraux_--_Cathédrale_Notre-Dame
Produits_de_luxe_--_Industrie_et_commerce,"[-0.023503052691618603, 0.051954121639331184, ...",Produits_de_luxe_--_Industrie_et_commerce
Éléments_(chimie)_--_Aspect_social,"[-0.08405434340238571, 0.08108076453208923, -0...",Éléments_(chimie)_--_Aspect_social
Gaspillage_des_finances_publiques,"[-0.08616872504353523, 0.03471718728542328, 0....",Gaspillage_des_finances_publiques
Assistance_médicalisée_pour_mourir_--_Aspect_religieux_--_Église_catholique,"[-0.0715772993862629, 0.10570614784955978, -0....",Assistance_médicalisée_pour_mourir_--_Aspect_r...
Responsabilité_(droit)_--_Histoire,"[-0.07808930426836014, 0.0645984634757042, -0....",Responsabilité_(droit)_--_Histoire
Capitalistes_et_financiers,"[-0.026554700220003724, 0.010577503100244535, ...",Capitalistes_et_financiers
Métaphore_--_Emploi_en_thérapeutique,"[0.023949647322297096, 0.06599561125040054, -0...",Métaphore_--_Emploi_en_thérapeutique


In [263]:
label_vectors.head(100)

,content_vector,label
RAMEAU,,
#c#Ndrangheta,"[-0.0745735118786494, 0.03379224302868048, -0....",#c#Ndrangheta
"#c#Ūd,_Musique_d#c#_--_Histoire","[-0.004272089339792728, 0.10384338349103928, -...","#c#Ūd,_Musique_d#c#_--_Histoire"
#c#Ūd_--_Facture_(instruments_de_musique),"[-0.04093937948346138, -0.07384087890386581, -...",#c#Ūd_--_Facture_(instruments_de_musique)
#c#Ūd_--_Histoire,"[-0.004272089339792728, 0.10384338349103928, -...",#c#Ūd_--_Histoire
"10_et_11_janvier_2015,_Manifestations_des_(France)","[-0.030750181525945663, 0.04315360262989998, -...","10_et_11_janvier_2015,_Manifestations_des_(Fra..."
...,...,...
Abolitionnistes,"[-0.040964093059301376, 0.02886986406520009, -...",Abolitionnistes
Abolitionnistes_--_Histoire,"[-0.06544297188520432, 0.015964774414896965, 0...",Abolitionnistes_--_Histoire
Abondance_(économie_politique),"[-0.01768941804766655, -0.0038525641430169344,...",Abondance_(économie_politique)


In [231]:
#sauvegarde des vecteur car long a produire
import pickle
label_vectors.to_pickle('cb_test_class_v2_vector_label.pkl') 
#label_vectors = pd.read_pickle('cb_test_class_v2_vector_label.pkl')

In [234]:
print(label_vectors.size)



125774


In [237]:
#alimentation index picone
from tqdm.auto import tqdm

# we will use batches of 256
batch_size = 256

for i in tqdm(range(0, len(label_vectors), batch_size)):
    # find end of batch
    i_end = min(i+batch_size, len(label_vectors))
    # extract batch
    batch = label_vectors.iloc[i:i_end]
    # select embeddings for batch
    emb = batch["content_vector"].tolist()
    # get metadata
    meta = [{"label": l} for l in batch["label"]]
    # create unique IDs
    ids = [f"{idx}" for idx in range(i, i_end)]
    # add all to upsert list
    to_upsert = list(zip(ids, emb, meta))
    # upsert/insert these records to pinecone
    _ = index.upsert(vectors=to_upsert)

  0%|          | 0/246 [00:00<?, ?it/s]

In [264]:
to_upsert

[('62720',
  [-0.0031574079766869545,
   -0.012853239042063555,
   0.027751542627811432,
   -0.07608327642083168,
   -0.0391421044866244,
   0.053137154007951416,
   -0.04282557281355063,
   0.08590315033992131,
   0.011098879699905714,
   0.015466515285273394,
   0.036108094888428845,
   -0.038559804670512676,
   0.03921550636490186,
   -0.04196954766909281,
   -0.04757632804103196,
   -0.07128716136018436,
   -0.03250103071331978,
   -0.04704896826297045,
   0.013209367791811625,
   0.03447853277126948,
   0.04114846015969912,
   -0.04945349941651026,
   0.024013972530762356,
   0.0108071972305576,
   0.0011015764127175014,
   -0.05028153335054716,
   0.004803851246833801,
   -0.021727894976114232,
   -0.02222566958516836,
   -0.022144639030254137,
   0.013772429122279087,
   0.029951420923074085,
   -0.0004725906377037366,
   0.010927237880726656,
   0.054731860756874084,
   -0.0016612131148576736,
   0.07650242000818253,
   0.002007116253177325,
   -0.0032632118090987206,
   0.0923

In [238]:
# check that we have all vectors in index
index.describe_index_stats()

{'dimension': 384,
 'index_fullness': 0.0,
 'namespaces': {'': {'vector_count': 62887}},
 'total_vector_count': 62887}

In [271]:
from pprint import pprint
def select_test_article(index):
    print("choisi un element du fichier de  test:")
    # select the article associated with index from test split
    article = df_test.iloc[index]
    # print test article data
    data = {"RESUME": article.RESUME[:1000],  "Original Labels": list(article.RAMEAU)}
    pprint(data)
    return article
def query_pinecone(article, top_k=3):
    print("predict rameau:")
    # Create embeddings for test articles
    xq = model.encode(article.RESUME).tolist()
    # query pinecone for labels
    results = index.query(xq, top_k=top_k, include_metadata=True)
    print(results)
    # select only the labels from result and print
    labels = [res["metadata"]["label"] for res in results.matches]
    pprint({"Predicted Labels": labels})

In [ ]:
#test sur le sixieme record du jeu de test
article = select_test_article(0)

In [ ]:
#interrogation db vectorized pour ce record
query_pinecone(article, top_k=10)

In [268]:
#generation fichier excel sur evaluation

#import pandas as pd
import numpy as np
import simplemma
import nltk
from texthero import preprocessing as preprocessing2
import texthero as hero
import pickle
import pandas as pd
import numpy as np




def lemmatize_text(text):
    w_tokenizer = nltk.tokenize.WhitespaceTokenizer()
    return ' '.join(word for word in [simplemma.lemmatize(w, lang='fr') for w in w_tokenizer.tokenize(text)])

import pinecone

# connect to pinecone environment
pinecone.init(
    api_key="VOTRE_CLE_PINECONE",
    environment="us-west4-gcp"  # find next to API key in console
)

def predict(text):
    xq = model.encode(lemmatize_text(text)).tolist()
    results = index.query(xq, top_k=10, include_metadata=True)
    return results

#chargement model
from sentence_transformers import SentenceTransformer
import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'

# load the model from huggingface
model = SentenceTransformer(
    'sentence-transformers/all-MiniLM-L6-v2',
    device=device
)

index = pinecone.Index('extreme-ml')
# faut jour les entetes et le connection a la base vecteurs
#df1 = pd.read_csv('export_picone_100.csv', sep='\t', skiprows=[12635,14570,20335,20655,32592,32975,32976,33310,50886,58789,58841,59676,63232,67341,72847,79427,80517,88343])
df1=testing_data.head(100).copy()
#df1=testing_data.copy()
df1["RESUME"] = df1['TITRE'].astype(str) +", "+ df1["RESUME"]
#-df=df1.drop(columns=['PPN','TITRE','DEWEY'])
df_test=df1
df_test['RAMEAU']=df_test['RAMEAU'].str.replace('_','#u#', regex=True).replace('"','#d#', regex=True).replace('\'','#c#', regex=True).replace(' ','_', regex=True)
#tronque
df_test['RESUME'] = df_test['RESUME'].str.slice(0,1000)
w_tokenizer = nltk.tokenize.WhitespaceTokenizer()
lemmatizer = nltk.stem.WordNetLemmatizer()
#lemmatization
df_test['RESUME'] = df_test['RESUME'].apply(lemmatize_text)
df_test['RAMEAU'] = df_test['RAMEAU'].str.split(';')
#df_test=df.copy()
#df_test.drop(index=df_test.index[:89200],axis=0, inplace=True)
#df_test.reset_index(drop=True, inplace=True)
#predictions
df_test['PREDICT'] = df_test['RESUME'].apply(predict)
#df_test['PREDICT'].head()
#df_test.to_csv('test_cbd.csv', sep='\t', encoding='utf-8')
df_test.to_csv('test_cbd_100_2.csv',index=False)
df_test

,CLE,PPN,TITRE,RESUME,RAMEAU,DEWEY,DESCR,presence_chaine_indexation,rameau_chaines_index,rameau_concepts,PREDICT
0,3,00002564X,La révolution structurale,"le révolution structurale, mutation ou crises,...",[Structuralisme],100,"La révolution structurale Mutations ou crises,...",False,['Structuralisme'],['Structuralisme'],"{'matches': [{'id': '29148', 'me..."
1,17,000057525,La théorie des jeux et ses applications à l'éc...,le théorie un jeu et son application à l'écono...,"[Mathématiques_économiques, Théorie_des_jeux]",519.3,La théorie des jeux et ses applications à l'éc...,False,"['Mathématiques économiques', 'Théorie des jeux']","['Mathématiques économiques', 'Théorie des jeux']","{'matches': [{'id': '34718', 'me..."
2,22,000079472,Pétrole : le vrai dossier,"pétrole : le vrai dossier, que dissimuler le é...",[Crise_économique_(1973)],320,Pétrole : le vrai dossier Que dissimulent les ...,False,['Crise économique (1973)'],['Crise économique (1973)'],"{'matches': [{'id': '46826', 'me..."
3,23,000085642,Magie : aspects de la tradition occidentale,"magie : aspect de le tradition occidentale, le...",[Magie],100,Magie : aspects de la tradition occidentale La...,False,['Magie'],['Magie'],"{'matches': [{'id': '33494', 'me..."
4,25,000087637,Mathématiques de base pour les linguistes,"mathématique de base pour le linguistes, le fo...","[Linguistique_--_Méthodes_graphiques, Linguist...",510,Mathématiques de base pour les linguistes La f...,True,"['Linguistique -- Méthodes graphiques', 'Lingu...","['Linguistique', 'Méthodes graphiques', 'Lingu...","{'matches': [{'id': '23295', 'me..."
...,...,...,...,...,...,...,...,...,...,...,...
95,479,001246259,Moïse géographe : recherches sur les représent...,moïse géographe : recherche sur le représentat...,"[Espace_--_Aspect_religieux_--_Christianisme, ...",200,Moïse géographe : recherches sur les représent...,True,['Espace -- Aspect religieux -- Christianisme'...,"['Espace', 'Aspect religieux', 'Christianisme'...","{'matches': [{'id': '25857', 'me..."
96,481,001249835,Former des enfants producteurs de textes,"former un enfant producteur de textes, groupe ...","[Créativité_(éducation), Français_(langue)_--_...",370,Former des enfants producteurs de textes Group...,True,"['Créativité (éducation)', 'Français (langue) ...","['Créativité (éducation)', 'Français (langue)'...","{'matches': [{'id': '23263', 'me..."
97,486,001263099,Multilinguisme et multiculturalisme en Amériqu...,multilinguisme et multiculturalisme en Amériqu...,"[Femmes_et_judaïsme, Littérature_et_société, L...",801,Multilinguisme et multiculturalisme en Amériqu...,False,"['Femmes et judaïsme', 'Littérature et société...","['Femmes et judaïsme', 'Littérature et société...","{'matches': [{'id': '32428', 'me..."
98,492,001278916,Aspects du mythe,"aspect de le mythe, le fonction de le mythe êt...","[Mythe, Mythologie_comparée, Société_primitive]",100,Aspects du mythe La fonction du mythe est de d...,False,"['Mythe', 'Mythologie comparée', 'Société prim...","['Mythe', 'Mythologie comparée', 'Société prim...","{'matches': [{'id': '37027', 'me..."


In [267]:
import json
import csv
import pandas
import ast

df_res=pd.DataFrame()

def extract_json(ppn,text):
    #text = ast.literal_eval(json.dumps(text))
    global df_res
    #print(text)
    x = json.loads(text)
    load_items = []
    
    for item in x["matches"]:
        #print(item["metadata"]["label"])
        load_items.append({'ppn': ppn,'score': item['score'],'label': item["metadata"]["label"]  })
            #load_items.append({'metadata': item2['label'] })
    z = pd.DataFrame(load_items)
    df_res=df_res.append(load_items, ignore_index=True)
    return z
df_test2 = df_test[['PPN', 'PREDICT']]
df_test2['PREDICT'] = df_test2['PREDICT'].astype(str).replace('\'', '"', regex=True).replace('\n', '', regex=True).replace('             ', ' ', regex=True)
#df_test2['PREDICT'] = df_test2['PREDICT'].astype(str).replace('""', '#db#', regex=True).replace('"', '#dc#', regex=True).replace('#db#', '\'', regex=True).replace('#dc#', '\'', regex=True).replace('\"', '#df#', regex=True).replace('\'', '"', regex=True).replace('\n', '', regex=True).replace('             ', ' ', regex=True)



df_test2.apply(lambda x: extract_json(x.PPN, x.PREDICT), axis=1)
df_res['label']=df_res['label'].replace('#u#','_', regex=True).replace('#d#','"', regex=True).replace('#c#','\'', regex=True).replace('_',' ', regex=True)
df_res.to_csv('test_cbd_100_3.csv',index=False)

  

/tmp/ipykernel_54440/1872417491.py:23: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_test2['PREDICT'] = df_test2['PREDICT'].astype(str).replace('\'', '"', regex=True).replace('\n', '', regex=True).replace('             ', ' ', regex=True)
/tmp/ipykernel_54440/1872417491.py:20: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  df_res=df_res.append(load_items, ignore_index=True)
/tmp/ipykernel_54440/1872417491.py:20: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  df_res=df_res.append(load_items, ignore_index=True)
/tmp/ipykernel_54440/1872417491.py:20: FutureWarning: The fram

/tmp/ipykernel_54440/1872417491.py:20: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  df_res=df_res.append(load_items, ignore_index=True)
/tmp/ipykernel_54440/1872417491.py:20: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  df_res=df_res.append(load_items, ignore_index=True)
/tmp/ipykernel_54440/1872417491.py:20: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  df_res=df_res.append(load_items, ignore_index=True)
/tmp/ipykernel_54440/1872417491.py:20: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  df_res=df_res.append(load_items, ignore_index=True)
/tmp/ipykernel_54440/1872417491.py:20: FutureWarning: The frame.append method is deprecated and 

/tmp/ipykernel_54440/1872417491.py:20: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  df_res=df_res.append(load_items, ignore_index=True)
/tmp/ipykernel_54440/1872417491.py:20: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  df_res=df_res.append(load_items, ignore_index=True)
/tmp/ipykernel_54440/1872417491.py:20: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  df_res=df_res.append(load_items, ignore_index=True)
/tmp/ipykernel_54440/1872417491.py:20: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  df_res=df_res.append(load_items, ignore_index=True)
/tmp/ipykernel_54440/1872417491.py:20: FutureWarning: The frame.append method is deprecated and 

In [251]:
df_res.head()

,ppn,score,label
0,00002564X,0.797656,Innovation_--_Philosophie
1,00002564X,0.779785,Innovation_--_Aspect_social
2,00002564X,0.776861,Technique_--_Aspect_social
3,00002564X,0.775053,Technique_et_civilisation
4,00002564X,0.774057,Prévision_technologique


In [14]:
# test sur une entree
# a jouer une fois
#import pandas as pd
import numpy as np
import simplemma
import nltk
from texthero import preprocessing as preprocessing2
import texthero as hero
import pickle



def lemmatize_text(text):
    w_tokenizer = nltk.tokenize.WhitespaceTokenizer()
    return ' '.join(word for word in [simplemma.lemmatize(w, lang='fr') for w in w_tokenizer.tokenize(text)])

import pinecone

# connect to pinecone environment
pinecone.init(
    api_key="VOTRE_CLE_PINECONE",
    environment="us-west4-gcp"  # find next to API key in console
)

def predict(text):
    xq = model.encode(lemmatize_text(text)).tolist()
    results = index.query(xq, top_k=100, include_metadata=True)
    return results

#chargement model
from sentence_transformers import SentenceTransformer
import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'

# load the model from huggingface
model = SentenceTransformer(
    'sentence-transformers/all-MiniLM-L6-v2',
    device=device
)

index = pinecone.Index('extreme-ml')




In [18]:

predict('le dessin être un voyage : carnet de jean Léonard, le dizaine de carnet de voyages, rempli à le gré un années, temoignent de l inlassable curiosité et de le gourmandise intellectuel avec  laquelles jean Léonard aura retenir le beauté de le monde à le  pointe de le crayon. pour qui être passionné par le dessin,  feuilleter ce centaine de page être un contentement sans égal. y   être fixé le ligne un paysage contemplés, le ombre un ruer  arpentées, le contour un architecture visitées. rempart   d Essaouira, grand muraille de Chine, jardin de le Daisen-In à  kyoto, temple de Karnak, Skyline de New-York, Palais un filateur à  Ahmedabad, ruiner de Mycènes, panthéon romain, bord de mer en  bretagne...')

{'matches': [{'id': '7129',
              'metadata': {'label': 'Carnets_de_croquis'},
              'score': 0.77722007,
              'values': []},
             {'id': '29883',
              'metadata': {'label': 'Monuments_historiques'},
              'score': 0.755565524,
              'values': []},
             {'id': '2943',
              'metadata': {'label': 'Architecture_--_Histoire'},
              'score': 0.750746,
              'values': []},
             {'id': '29860',
              'metadata': {'label': 'Monuments'},
              'score': 0.745746791,
              'values': []},
             {'id': '49352',
              'metadata': {'label': 'Voyage'},
              'score': 0.74461633,
              'values': []},
             {'id': '34030',
              'metadata': {'label': 'Patrimoine_urbain'},
              'score': 0.744391799,
              'values': []},
             {'id': '13152',
              'metadata': {'label': 'Dessin_--_Thèmes,_motifs'},
        